# What should Ubisoft's next game be?

A market study of the **55 691 games** listed on Steam, run for Ubisoft's studio leadership.

Jedha *Full Stack Data Scientist* — **Block 2, Big Data Project**. PySpark on Databricks.

---

Ubisoft wants to launch a new game and asked for a global reading of the Steam marketplace
before the concept is locked. The brief lists a dozen questions on three levels — the market as
a whole, genres, and platforms. This notebook takes the three levels in that order, and each one
closes on the one thing it changes in the product brief: **genre, price, platforms, languages,
release window, age rating.**

Section 6 collects those six decisions. Section 7 says what this dataset cannot decide.

## Where each question is answered

| # | The brief asks | Answered in |
|---|---|---|
| | ***Macro*** | |
| 1 | Which publisher has released the most games on Steam? | 3.1 |
| 2 | What are the best rated games? | **3.6** |
| 3 | Are there years with more releases? More or fewer during Covid? | 3.2 |
| 4 | How are the prices distributed? Are there many games with a discount? | 3.3 |
| 5 | What are the most represented languages? | 3.4 |
| 6 | Are there many games prohibited for children under 16/18? | 3.5 |
| | ***Genres*** | |
| 7 | What are the most represented genres? | 4.1 |
| 8 | Are there any genres with a better positive/negative review ratio? | 4.2 |
| 9 | Do some publishers have favourite genres? | 4.3 |
| 10 | What are the most lucrative genres? | 4.4 |
| | ***Platforms*** | |
| 11 | Are most games available on Windows/Mac/Linux? | 5.1 |
| 12 | Do certain genres tend to be available on certain platforms? | 5.2 |

Question 2 closes the macro level instead of coming second: 3.2 to 3.5 each end on a product
decision, while the best-rated list produces a benchmark to hit rather than a choice to make.

Four sections go beyond the list, because the brief's stated goal — *what factors affect the
popularity or sales of a video game* — needs them. **2.11** defines what success means in a
dataset that holds no sales figure, **4.5** tests whether any genre is actually emerging, **4.6**
locates the genre-and-price slot, and **5.3** asks whether porting pays once price is held equal.

## The data

`steam_game_output.json` — a 61 MB JSON array, one object per game, `{"id": ..., "data": {...}}`,
served from `s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/`. It is a **SteamSpy
snapshot** and the newest release in it is **11 November 2022**, so every count for 2022 is
partial and no game released after that date exists here.

Three of the 22 fields are nested — `tags` (a tag → number-of-votes object), `platforms`
(three booleans) and `categories` (a list) — which is what makes the file semi-structured and
what `explode()` and `getField()` are for.

## 0. Environment

The notebook is written for **Databricks**, and falls back to a local Spark session so the code
can be re-run outside a workspace. Everything below this cell is identical in both cases,
including the `display()` calls that drive Databricks' visualisation tool.

Two things differ per environment and are hidden behind the same names: `display()`, and
`materialise()` — serverless compute has no `cache()`, so a frame that many later cells re-read
is written to a Delta table there, and simply cached locally.

In [1]:
import os

IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ
SOURCE_URL = ("https://full-stack-bigdata-datasets.s3.amazonaws.com"
              "/Big_Data/Project_Steam/steam_game_output.json")

if IS_DATABRICKS:
    # Unity Catalog volume: the file is fetched once, then read from storage.
    spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.steam")
    DATA_PATH = "/Volumes/workspace/default/steam/steam_game_output.json"
    if not os.path.exists(DATA_PATH):
        import urllib.request
        urllib.request.urlretrieve(SOURCE_URL, DATA_PATH)

    def materialise(df, name):
        # No cache() on serverless. A Delta table costs one write and turns the
        # non-splittable JSON into columnar storage for every read that follows.
        table = f"workspace.default.{name}"
        df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
        return spark.table(table)
else:
    # Local run: build the session by hand and emulate Databricks' display().
    from pyspark.sql import SparkSession, DataFrame
    from IPython.display import display as _ipython_display

    os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
    spark = (SparkSession.builder.appName("steam")
             .master("local[*]")
             .config("spark.driver.memory", "10g")
             .config("spark.sql.session.timeZone", "UTC")
             .config("spark.ui.showConsoleProgress", "false")
             .getOrCreate())
    spark.sparkContext.setLogLevel("ERROR")

    def display(x, n=1000):
        _ipython_display(x.limit(n).toPandas() if isinstance(x, DataFrame) else x)

    def materialise(df, name):
        return df.cache()

    DATA_PATH = "data/steam_game_output.json"

from pyspark.sql import functions as F, Window
from pyspark.sql.types import (StructType, StructField, StringType, LongType,
                               BooleanType, ArrayType, MapType)

print("Spark", spark.version, "| Databricks" if IS_DATABRICKS else "| local", "|", DATA_PATH)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 3.5.3 | local | data/steam_game_output.json


## 1. Reading a semi-structured file

The whole array sits on **one line**, so `multiLine` has to be on: without it Spark splits the
file on newlines and finds a single unparseable record.

### 1.1 Why schema inference is not enough

Let Spark infer the schema and `tags` — an object whose *keys are the tag names* — becomes a
struct with one column per tag seen anywhere in the file. Inference also has to scan the 61 MB
twice, and it silently picks a type for `required_age`, a field that holds both numbers and
strings.

In [2]:
inferred = spark.read.option("multiLine", True).json(DATA_PATH)

tags_field = inferred.schema["data"].dataType["tags"].dataType
print("columns Spark invented for `tags`:", len(tags_field.fields))
print("first five:", [f.name for f in tags_field.fields[:5]])

columns Spark invented for `tags`: 441
first five: ['1980s', "1990's", '2.5D', '2D', '2D Fighter']


### 1.2 An explicit schema instead

Declaring `tags` as `MAP<STRING, BIGINT>` turns those 441 phantom columns into one map column
that `explode()` can open into (tag, votes) rows. `required_age` is declared `STRING` and parsed
later, where the parsing rule is visible.

In [3]:
DATA_SCHEMA = StructType([
    StructField("appid", LongType()),
    StructField("name", StringType()),
    StructField("short_description", StringType()),
    StructField("developer", StringType()),
    StructField("publisher", StringType()),
    StructField("genre", StringType()),                       # comma-separated
    StructField("tags", MapType(StringType(), LongType())),   # tag -> community votes
    StructField("type", StringType()),
    StructField("categories", ArrayType(StringType())),
    StructField("owners", StringType()),                      # bucketed range, e.g. "0 .. 20,000"
    StructField("positive", LongType()),
    StructField("negative", LongType()),
    StructField("price", StringType()),                       # US cents, as a string
    StructField("initialprice", StringType()),
    StructField("discount", StringType()),                    # percent, as a string
    StructField("ccu", LongType()),                           # peak concurrent users
    StructField("languages", StringType()),                   # comma-separated
    StructField("platforms", StructType([
        StructField("windows", BooleanType()),
        StructField("mac", BooleanType()),
        StructField("linux", BooleanType()),
    ])),
    StructField("release_date", StringType()),
    StructField("required_age", StringType()),
    StructField("website", StringType()),
    StructField("header_image", StringType()),
])

SCHEMA = StructType([
    StructField("id", StringType()),
    StructField("data", DATA_SCHEMA),
])

raw = (spark.read.schema(SCHEMA).option("multiLine", True).json(DATA_PATH)
       .select("id", "data.*"))          # flatten the nested `data` struct

print(f"{raw.count():,} games x {len(raw.columns)} columns")
raw.printSchema()

55,691 games x 23 columns
root
 |-- id: string (nullable = true)
 |-- appid: long (nullable = true)
 |-- name: string (nullable = true)
 |-- short_description: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- tags: map (nullable = true)
 |    |-- key: string
 |    |-- value: long (valueContainsNull = true)
 |-- type: string (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- owners: string (nullable = true)
 |-- positive: long (nullable = true)
 |-- negative: long (nullable = true)
 |-- price: string (nullable = true)
 |-- initialprice: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- ccu: long (nullable = true)
 |-- languages: string (nullable = true)
 |-- platforms: struct (nullable = true)
 |    |-- windows: boolean (nullable = true)
 |    |-- mac: boolean (nullable = true)
 |    |-- linux: boolean (nulla

### 1.3 The map opens

`explode()` on the `tags` map is what gives every later tag question a row-per-tag table.

In [4]:
display(
    raw.filter(F.col("name") == "Counter-Strike")
       .select("name", F.explode("tags").alias("tag", "votes"))
       .orderBy(F.desc("votes")).limit(8)
)

,name,tag,votes
0,Counter-Strike,Action,5426
1,Counter-Strike,FPS,4831
2,Counter-Strike,Multiplayer,3392
3,Counter-Strike,Shooter,3353
4,Counter-Strike,Classic,2784
5,Counter-Strike,Team-Based,1864
6,Counter-Strike,First-Person,1707
7,Counter-Strike,Competitive,1607


## 2. Cleaning

Nine rules. Each one is stated with the number of rows it touches, so nothing is silently
dropped or rewritten.

### 2.1 Keys and scope

`id` duplicates `appid`, and every `appid` is unique — there is nothing to deduplicate. One row
is not a game.

In [5]:
print("rows where id != appid:", raw.filter(F.col("id") != F.col("appid").cast("string")).count())
print("distinct appid:", raw.select("appid").distinct().count(), "of", raw.count(), "rows")
display(raw.groupBy("type").count())

rows where id != appid: 0


distinct appid: 55691 of 55691 rows


,type,count
0,hardware,1
1,game,55690


### 2.2 Money

`price`, `initialprice` and `discount` are strings. Prices are US cents, so a `999` is $9.99.
`price` is the current price and `initialprice` the list price: the two differ on exactly the
2 518 rows that carry a discount, so the field is internally consistent and `initialprice` is
the one to use for anything about a game's positioning.

In [6]:
games = (raw.filter(F.col("type") == "game")
         .withColumn("price_usd", F.col("price").cast("int") / 100)
         .withColumn("initial_price_usd", F.col("initialprice").cast("int") / 100)
         .withColumn("discount_pct", F.col("discount").cast("int"))
         .withColumn("is_free", F.col("price").cast("int") == 0))

display(games.select(
    F.round(F.min("price_usd"), 2).alias("min_price"),
    F.round(F.max("price_usd"), 2).alias("max_price"),
    F.sum(F.col("is_free").cast("int")).alias("free_games"),
    F.sum((F.col("discount_pct") > 0).cast("int")).alias("discounted"),
    F.sum((F.col("price") != F.col("initialprice")).cast("int")).alias("price_below_list"),
))

,min_price,max_price,free_games,discounted,price_below_list
0,0.0,999.0,7779,2518,2518


### 2.3 Release dates

Four shapes in one column: `2020/06/18`, `2020/06/8` (one-digit day), `2015/09` (no day at all)
and empty. Matching each shape before parsing keeps the 99 empty strings as an explicit `NULL`
instead of a silent failure.

In [7]:
games = (games
    .withColumn("release_date_parsed",
        F.when(F.col("release_date").rlike(r"^\d{4}/\d{1,2}/\d{1,2}$"),
               F.to_date("release_date", "yyyy/M/d"))
         .when(F.col("release_date").rlike(r"^\d{4}/\d{1,2}$"),
               F.to_date("release_date", "yyyy/M")))
    .withColumn("release_year", F.year("release_date_parsed"))
    .withColumn("release_month", F.month("release_date_parsed")))

display(games.select(
    F.min("release_date_parsed").alias("first_release"),
    F.max("release_date_parsed").alias("last_release"),
    F.sum(F.col("release_date_parsed").isNull().cast("int")).alias("unparseable"),
))

,first_release,last_release,unparseable
0,1997-06-30,2022-11-11,99


### 2.4 Comma-separated lists

`genre` and `languages` pack several values into one string. Splitting on the comma is not quite
enough: four rows carry a parenthetical note (`English (full audio)`), two a stray semicolon,
and one lists English twice. Stripping the noise, trimming and deduplicating gives clean arrays —
and note that `Spanish - Spain` and `Design & Illustration` mean the separator can only ever be
the comma.

In [8]:
def split_list(column):
    """Comma-separated string -> trimmed, deduplicated array, parenthetical notes removed."""
    parts = F.split(F.regexp_replace(F.col(column), r"\([^)]*\)|[;*]", ""), ",")
    return F.array_distinct(F.array_remove(F.transform(parts, lambda x: F.trim(x)), ""))

games = (games
    .withColumn("genres", split_list("genre"))
    .withColumn("languages_list", split_list("languages"))
    .withColumn("n_genres", F.size("genres"))
    .withColumn("n_languages", F.size("languages_list")))

display(games.filter(F.col("languages").contains("(")).select("name", "languages", "languages_list"))

,name,languages,languages_list
0,Ninja Reflex: Steamworks Edition,"English, French, German, Italian, Spanish - Sp...","[English, French, German, Italian, Spanish - S..."
1,The Witcher: Enhanced Edition Director's Cut,"English, French, German, Spanish - Spain, Ital...","[English, French, German, Spanish - Spain, Ita..."
2,Wallace & Gromit’s Grand Adventures,"English (full audio), French, German, Italian,...","[English, French, German, Italian, Spanish - S..."
3,Marble Masters: The Pit,"English, French, German, Spanish - Spain, Czec...","[English, French, German, Spanish - Spain, Cze..."


### 2.5 Publisher names

Ubisoft appears under five spellings, two of which differ only by a trademark sign or four
trailing tabs. Trimming whitespace and stripping `®`/`™` rewrites 271 rows and empties 134
more that held nothing but whitespace — 405 changes in all. The report in 2.10 therefore
compares with `eqNullSafe` and not `!=`, or SQL's `NULL != 'x' -> NULL` would silently drop
those 134. It does **not** merge `Ubisoft` with `Ubisoft Entertainment`, and it should not —
deciding that two different company names are the same firm is a judgement call, not a cleaning
rule. The publisher counts in section 3.1 are therefore a floor, not an exact figure.

The field also holds co-publisher lists (`Team17, NEXT Studios`), but 3 215 rows contain a comma
and most of them are `Ltd.`-style suffixes, so splitting on it would create more noise than it
removes. The string is kept whole.

In [9]:
def normalise_name(column):
    cleaned = F.trim(F.regexp_replace(F.regexp_replace(F.col(column), r"[®™]", ""), r"\s+", " "))
    return F.when(cleaned != "", cleaned)

games = games.withColumn("publisher_clean", normalise_name("publisher"))

print("distinct publisher strings:", games.select("publisher").distinct().count(),
      "-> after normalisation:", games.select("publisher_clean").distinct().count())

display(games.filter(F.col("publisher").rlike("^Ubisoft"))
             .groupBy("publisher", "publisher_clean").count().orderBy(F.desc("count")))

distinct publisher strings: 29966 -> after normalisation: 29825


,publisher,publisher_clean,count
0,Ubisoft,Ubisoft,127
1,Ubisoft Entertainment,Ubisoft Entertainment,5
2,Ubisoft Entertainment\t\t\t\t,Ubisoft Entertainment,1
3,Ubisoft®,Ubisoft,1
4,Ubisoft - San Francisco,Ubisoft - San Francisco,1


### 2.6 Owners

SteamSpy does not publish a sales figure, it publishes a bracket: `"10,000,000 .. 20,000,000"`.
Two regexes give the bounds, and the midpoint stands in for the value. **68% of the catalogue
sits in the bottom bracket**, so the median of that midpoint is `10,000` almost everywhere and is
useless as a comparison — the rest of the notebook uses two other measures instead: the **median
review count**, and the **share of games above 100 000 owners** (the "breakout rate").

In [10]:
owners_digits = F.regexp_replace(F.col("owners"), ",", "")
games = (games
    .withColumn("owners_min", F.regexp_extract(owners_digits, r"^(\d+)", 1).cast("long"))
    .withColumn("owners_max", F.regexp_extract(owners_digits, r"\.\.\s*(\d+)$", 1).cast("long")))
games = games.withColumn("owners_mid", (F.col("owners_min") + F.col("owners_max")) / 2)

display(games.groupBy("owners", "owners_min", "owners_max").count().orderBy("owners_min"))

,owners,owners_min,owners_max,count
0,"0 .. 20,000",0,20000,38072
1,"20,000 .. 50,000",20000,50000,7285
2,"50,000 .. 100,000",50000,100000,3695
3,"100,000 .. 200,000",100000,200000,2519
4,"200,000 .. 500,000",200000,500000,2162
5,"500,000 .. 1,000,000",500000,1000000,932
6,"1,000,000 .. 2,000,000",1000000,2000000,526
7,"2,000,000 .. 5,000,000",2000000,5000000,335
8,"5,000,000 .. 10,000,000",5000000,10000000,97
9,"10,000,000 .. 20,000,000",10000000,20000000,41


### 2.7 Required age

The field mixes integers and strings, and its odd values are `"MA 15+"`, `"21+"`, `"7+"`, `"35"`
and `"180"` (four times). Pulling the first number out handles the `+` suffixes; anything outside
0-21 is not an age rating and becomes `NULL` — five rows.

That fixes the parsing but not the field: **it is 0 for 98.8% of the catalogue**, because Steam
gates mature content through its own content descriptors rather than this legacy attribute.
Section 3.5 answers the brief's question about age-restricted games from the community tags
instead.

In [11]:
age = F.regexp_extract(F.col("required_age"), r"(\d+)", 1).cast("int")
games = games.withColumn("age_rating", F.when(age.between(0, 21), age))

display(games.groupBy("required_age", "age_rating").count().orderBy(F.desc("count")).limit(25))

,required_age,age_rating,count
0,0,0.0,55029
1,15,15.0,264
2,18,18.0,223
3,17,17.0,38
4,16,16.0,38
5,12,12.0,32
6,13,13.0,26
7,14,14.0,10
8,10,10.0,7
9,6,6.0,4


### 2.8 Reviews

`positive` and `negative` are review counts, not scores. The raw share of positive reviews is
unusable as a ranking on its own: **8 634 games sit at exactly 100%**, most of them on a handful
of reviews. The Wilson 95% lower bound answers the question actually being asked — *how good is
this game, given how little we know about it* — by pulling small samples towards the middle.

In [12]:
games = (games
    .withColumn("reviews", F.col("positive") + F.col("negative"))
    .withColumn("positive_ratio",
                F.when(F.col("positive") + F.col("negative") > 0,
                       F.col("positive") / (F.col("positive") + F.col("negative")))))

z, n, p = F.lit(1.96), F.col("reviews"), F.col("positive_ratio")
wilson_lower_bound = (p + z * z / (2 * n) - z * F.sqrt((p * (1 - p) + z * z / (4 * n)) / n)) / (1 + z * z / n)
games = games.withColumn("wilson_score", F.when(n > 0, wilson_lower_bound))

display(games.filter(F.col("positive_ratio") == 1)
             .select("name", "positive", "negative", "positive_ratio", "wilson_score")
             .orderBy("reviews").limit(5))

,name,positive,negative,positive_ratio,wilson_score
0,CrossTrix,1,0,1.0,0.206543
1,Anti-Grav Bamboo-copter,1,0,1.0,0.206543
2,De Profundis,1,0,1.0,0.206543
3,Kill Tiger,1,0,1.0,0.206543
4,The Truck Game,1,0,1.0,0.206543


### 2.9 Platforms, and a value proxy

The `platforms` struct becomes three flags and a count. Revenue is not in the dataset, and
nothing here stands in for it: `owners` counts copies *held*, not copies bought — bundles, gift
keys and free weekends all land in it — and `initialprice` is the list price, not a price anyone
paid. Their product is named for what it is, **`owners_x_price`: the list-price value of every
copy in circulation**. The rankings in sections 3 and 4 use it as a sort key; section 7 takes it
apart.

In [13]:
games = (games
    .withColumn("windows", F.col("platforms.windows"))
    .withColumn("mac", F.col("platforms.mac"))
    .withColumn("linux", F.col("platforms.linux"))
    .withColumn("n_platforms", F.col("platforms.windows").cast("int")
                             + F.col("platforms.mac").cast("int")
                             + F.col("platforms.linux").cast("int"))
    .withColumn("owners_x_price", F.col("owners_mid") * F.col("initial_price_usd")))

games = materialise(games, "steam_games")
print(f"{games.count():,} games ready, {len(games.columns)} columns")

55,690 games ready, 47 columns


### 2.10 Cleaning report

In [14]:
report = spark.createDataFrame([
    ("2.1   drop type != 'game'",           games.count(), "rows dropped",
     raw.count() - games.count()),
    ("2.2   price strings -> USD",          games.count(), "non-numeric source",
     games.filter(~F.col("price").rlike(r"^-?\d+$")
                | ~F.col("initialprice").rlike(r"^-?\d+$")
                | ~F.col("discount").rlike(r"^-?\d+$")).count()),
    ("2.3   release_date -> date",          games.count(), "source empty -> NULL",
     games.filter(F.col("release_date_parsed").isNull()).count()),
    ("2.4a  genre -> array",                games.count(), "source empty -> []",
     games.filter(F.col("n_genres") == 0).count()),
    ("2.4b  languages -> array",            games.count(), "source empty -> []",
     games.filter(F.col("n_languages") == 0).count()),
    ("2.5   publisher name normalised",     games.count(), "values rewritten",
     games.filter(~F.col("publisher").eqNullSafe(F.col("publisher_clean"))).count()),
    ("2.6   owners bracket -> bounds",      games.count(), "unparsed -> NULL",
     games.filter(F.col("owners_min").isNull()).count()),
    ("2.7   required_age -> 0-21 or NULL",  games.count(), "out of range -> NULL",
     games.filter(F.col("age_rating").isNull()).count()),
    ("2.8   reviews -> Wilson score",       games.count(), "no reviews -> NULL",
     games.filter(F.col("wilson_score").isNull()).count()),
], ["rule", "rows_in", "measured", "n"])

display(report)

,rule,rows_in,measured,n
0,2.1 drop type != 'game',55690,rows dropped,1
1,2.2 price strings -> USD,55690,non-numeric source,0
2,2.3 release_date -> date,55690,source empty -> NULL,99
3,2.4a genre -> array,55690,source empty -> [],160
4,2.4b languages -> array,55690,source empty -> [],10
5,2.5 publisher name normalised,55690,values rewritten,405
6,2.6 owners bracket -> bounds,55690,unparsed -> NULL,0
7,2.7 required_age -> 0-21 or NULL,55690,out of range -> NULL,5
8,2.8 reviews -> Wilson score,55690,no reviews -> NULL,163


---
# 3. The market

## 3.1 Who publishes on Steam

*Chart: bar, `publisher_clean` x `games`.*

In [15]:
by_publisher = (games.groupBy("publisher_clean")
    .agg(F.count("*").alias("games"))
    .filter(F.col("publisher_clean").isNotNull()))

display(by_publisher.orderBy(F.desc("games")).limit(20))

,publisher_clean,games
0,Big Fish Games,423
1,8floor,202
2,SEGA,165
3,Strategy First,151
4,Square Enix,141
5,Choice of Games,140
6,Sekai Project,132
7,HH-Games,132
8,Ubisoft,128
9,Laush Studio,126


**Big Fish Games** has released the most games — 423, casual hidden-object titles — ahead of
8floor (202) and SEGA (165). Ubisoft is tenth with 128.

That ranking says less than the shape behind it. The 55 690 games spread over 29 824 named
publishers — 134 games carry no publisher at all, and appear in no ranking here — and the
concentration is the finding:

In [16]:
buckets = (by_publisher
    .withColumn("size", F.when(F.col("games") == 1, "1 game")
                         .when(F.col("games") <= 5, "2-5 games")
                         .when(F.col("games") <= 20, "6-20 games")
                         .otherwise("21+ games"))
    .groupBy("size").agg(F.count("*").alias("publishers"), F.sum("games").alias("games")))

# by_publisher drops the games with no publisher, so the buckets cover 55 556 of the 55 690.
# The residual row keeps the numerator on the same population as the denominator below.
orphans = (games.filter(F.col("publisher_clean").isNull())
    .groupBy(F.lit("no publisher").alias("size"))
    .agg(F.lit(0).cast("long").alias("publishers"), F.count("*").alias("games")))

concentration = (buckets.unionByName(orphans)
    .withColumn("pct_of_catalogue", F.round(100 * F.col("games") / games.count(), 1)))

# the total row holds the two catalogue-wide figures the text above quotes
total = concentration.agg(
    F.lit("all").alias("size"),
    F.sum("publishers").alias("publishers"),
    F.sum("games").alias("games"),
    F.round(100 * F.sum("games") / games.count(), 1).alias("pct_of_catalogue"))

display(concentration.unionByName(total)
        .orderBy(F.col("size").isin("no publisher", "all"), "publishers"))

,size,publishers,games,pct_of_catalogue
0,21+ games,195,9502,17.1
1,6-20 games,816,7984,14.3
2,2-5 games,5795,15052,27.0
3,1 game,23018,23018,41.3
4,no publisher,0,134,0.2
5,all,29824,55690,100.0


**41% of the catalogue comes from publishers that have released exactly one game**, and the
twenty largest publishers together account for 5% of releases. Steam is not a market of a few
big houses — it is a very long tail, and the 195 publishers with 21 releases or more hold 17%
of it.

## 3.2 The release calendar

*Chart: bar, `release_year` x `games`.*

In [17]:
per_year = (games.filter(F.col("release_year").isNotNull())
                 .groupBy("release_year").agg(F.count("*").alias("games"))
                 .orderBy("release_year"))
display(per_year)

,release_year,games
0,1997,2
1,1998,1
2,1999,3
3,2000,2
4,2001,4
5,2002,1
6,2003,3
7,2004,6
8,2005,6
9,2006,61


Two things are stacked in this column, and they separate around 2013. Before it the table counts
a curated storefront rather than a market: Steam only opened to third-party publishers in 2005
(Rag Doll Kung Fu and Darwinia, appid 1002 and 1500), and Valve picked by hand what went on sale
— 61 games in 2006, still only 471 in 2013. Greenlight opens the gate at the end of 2012 and
Steam Direct replaces it in 2017, and the two jumps in the table sit on that calendar: **471 to
1 557 in 2014**, then **4 185 to 6 017 in 2017**. Those two dates are Steam's history and not a
measurement from this file, but they are why the early years cannot be read as an industry
releasing fewer games. Those years are also a survivors' list — a November 2022 snapshot holds
only what was still on sale (section 7).

**Covid neither slowed releases nor set them off.** 7 678 in 2018, 6 968 in 2019, then 8 305 in
2020 and 8 823 in 2021: the only dip is the year *before* the pandemic, and 2020-2021 extends a
plateau that starts in 2018 rather than breaking it. 2022 reads 7 455, but `last_release` in 2.3
is 11 November 2022 — that year is ten and a half months long, and its fall is an artefact.

Inside the year, over the eight complete years from 2014, where the jump above turns the
catalogue into a market, to 2021, the last full year the snapshot covers:

*Chart: bar, `release_month` x `games`.*

In [18]:
display(games.filter(F.col("release_year").between(2014, 2021))
             .groupBy("release_month").agg(F.count("*").alias("games")).orderBy("release_month"))

,release_month,games
0,1,3096
1,2,3454
2,3,3742
3,4,3698
4,5,3809
5,6,3371
6,7,3903
7,8,4113
8,9,4196
9,10,4451


The calendar is not flat. **October is the busiest month at 4 451 releases and January the
quietest at 3 096**, 44% apart, and the shape is a season rather than noise: the six lightest
months of the year are exactly January to June, the six heaviest exactly July to December, with
a second trough in June. The second half is the run-up to the holiday sales, and it is where the
competition for a store slot sits.

The table counts competitors, not buyers — it says how many games a launch shares its month
with, and nothing about whether shipping next to them costs or pays.

**Decision — release window: the first half of the year, and not the September-December ramp.**

## 3.3 Price

*Chart: bar, `band` x `games`.*

In [19]:
# band_rank carries the ordering, so the labels can read as prices and nothing else
band_rank = (F.when(F.col("is_free"), 0)
              .when(F.col("price_usd") < 5, 1)
              .when(F.col("price_usd") < 10, 2)
              .when(F.col("price_usd") < 20, 3)
              .when(F.col("price_usd") < 40, 4)
              .otherwise(5))

price_band = (F.when(band_rank == 0, "free")
               .when(band_rank == 1, "under $5")
               .when(band_rank == 2, "$5-10")
               .when(band_rank == 3, "$10-20")
               .when(band_rank == 4, "$20-40")
               .otherwise("$40+"))

display(games.filter(~F.col("is_free")).select(
    F.round(F.min("price_usd"), 2).alias("min_paid"),
    F.round(F.percentile_approx("price_usd", 0.5), 2).alias("median_paid"),
    F.round(F.avg("price_usd"), 2).alias("mean_paid"),
    F.round(F.stddev("price_usd"), 2).alias("stddev_paid"),
    F.round(F.percentile_approx("price_usd", 0.9), 2).alias("p90"),
    F.round(F.percentile_approx("price_usd", 0.99), 2).alias("p99"),
    F.round(F.max("price_usd"), 2).alias("max_paid"),
    F.round(100 * F.avg(((F.col("price").cast("int") % 100) == 99).cast("int")), 1).alias("pct_ends_99")))

display(games.withColumn("rank", band_rank).withColumn("band", price_band)
        .groupBy("rank", "band").agg(
            F.count("*").alias("games"),
            F.round(100 * F.count("*") / games.count(), 1).alias("pct_games"))
        .orderBy("rank").drop("rank"))

,min_paid,median_paid,mean_paid,stddev_paid,p90,p99,max_paid,pct_ends_99
0,0.28,5.99,8.99,11.3,19.99,49.99,999.0,95.6


,band,games,pct_games
0,free,7779,14.0
1,under $5,23478,42.2
2,$5-10,12450,22.4
3,$10-20,9022,16.2
4,$20-40,2394,4.3
5,$40+,567,1.0


Steam is a cheap store, and it prices in `.99`: **95.6% of paid games end on those two digits**.
The mass sits at the bottom — **under $5 alone is 42.2% of the catalogue**, the largest band of
the six, and free or under $10 is 78.6% of it. The paid median is $5.99 against a mean of $8.99;
that gap, and a standard deviation of $11.30 on that mean, is a thin tail stretching right to a
$999 outlier. The 99th percentile is $49.99.

The second half of the question — *are there many games with a discount* — read on the same bands:

*Chart: bar, `band` x `pct_discounted`.*

In [20]:
display(games.filter(F.col("discount_pct") > 0).select(
    F.count("*").alias("discounted_games"),
    F.round(100 * F.count("*") / games.count(), 1).alias("pct_of_catalogue"),
    F.round(F.avg("discount_pct"), 1).alias("mean_discount"),
    F.percentile_approx("discount_pct", 0.5).alias("median_discount")))

# same bands as above, so the two tables read against each other
on_sale = F.col("discount_pct") > 0
display(games.withColumn("rank", band_rank).withColumn("band", price_band)
        .groupBy("rank", "band").agg(
            F.count("*").alias("games"),
            F.sum(on_sale.cast("int")).alias("discounted"),
            F.round(100 * F.avg(on_sale.cast("int")), 1).alias("pct_discounted"),
            F.round(F.avg(F.when(on_sale, F.col("discount_pct"))), 1).alias("mean_discount"))
        .orderBy("rank").drop("rank"))

,discounted_games,pct_of_catalogue,mean_discount,median_discount
0,2518,4.5,57.6,60


,band,games,discounted,pct_discounted,mean_discount
0,free,7779,0,0.0,NaN
1,under $5,23478,1884,8.0,63.6
2,$5-10,12450,380,3.1,45.0
3,$10-20,9022,211,2.3,31.6
4,$20-40,2394,40,1.7,33.7
5,$40+,567,3,0.5,24.7


**Not many: 2 518 games are on sale, 4.5% of the catalogue**, at a median 60% off. And the
discounting is not spread evenly across the store — **1 884 of those 2 518 are games under $5**,
the band that is already the largest. Rate and depth both fall with every step up: 8.0% of the
under-$5 games are discounted, at a mean 63.6% off, against 0.5% and 24.7% above $40. Cheap games
discount often and deep, expensive ones rarely and shallow.

This is a one-day snapshot and Steam's sales are periodic, so it measures the day the file was
pulled, not how often a game goes on sale. *Are there many games with a discount* has an answer;
*how often does a game go on sale* does not.

## 3.4 Languages

*Chart: bar, `language` x `games`.*

In [21]:
by_language = (games.select(F.explode("languages_list").alias("language"))
                    .groupBy("language").agg(F.count("*").alias("games")))

display(by_language
    .withColumn("pct_of_games", F.round(100 * F.col("games") / games.count(), 1))
    .orderBy(F.desc("games")).limit(20))

,language,games,pct_of_games
0,English,55116,99.0
1,German,14019,25.2
2,French,13426,24.1
3,Russian,12922,23.2
4,Simplified Chinese,12782,23.0
5,Spanish - Spain,12233,22.0
6,Japanese,10368,18.6
7,Italian,9304,16.7
8,Portuguese - Brazil,6750,12.1
9,Korean,6600,11.9


**English is on 99% of the catalogue** and is not a decision. The next tier is: German (14 019),
French (13 426), Russian (12 922), Simplified Chinese (12 782), Spanish (12 233), Japanese
(10 368), Italian (9 304). The real question is how far down that list to go.

*Chart: combo — bars `games`, line `breakout_pct`, on `languages`.*

In [22]:
def outcome(df, *group_by):
    """Volume and the two success measures, for any grouping."""
    return (df.groupBy(*group_by).agg(
        F.count("*").alias("games"),
        F.percentile_approx("reviews", 0.5).alias("median_reviews"),
        F.round(100 * F.avg((F.col("owners_min") >= 100000).cast("int")), 1).alias("breakout_pct"),
        F.round(F.avg("positive_ratio"), 3).alias("mean_positive_ratio")))

language_band = (F.when(F.col("n_languages") <= 1, "1")
                  .when(F.col("n_languages") <= 4, "2-4")
                  .when(F.col("n_languages") <= 9, "5-9")
                  .when(F.col("n_languages") <= 14, "10-14")
                  .when(F.col("n_languages") <= 20, "15-20")
                  .otherwise("21+"))

display(outcome(games.withColumn("languages", language_band), "languages")
        .orderBy(F.desc("median_reviews")))

,languages,games,median_reviews,breakout_pct,mean_positive_ratio
0,15-20,762,360,39.0,0.784
1,10-14,3595,306,34.7,0.773
2,5-9,7377,133,24.7,0.755
3,2-4,13026,26,7.9,0.741
4,21+,1265,25,13.0,0.747
5,1,29665,16,7.0,0.724


Localisation tracks success up to about twenty languages — 16 median reviews and a 7% breakout
rate at a single language, **360 reviews and 39% at 15-20 languages** — and then collapses at 21+.

That collapse mixes two things: how many languages a game ships, and what it costs — price being
the strongest predictor in this section. Holding the price band fixed separates them, the way 5.3
does for ports:

In [23]:
display(outcome(games.filter((F.col("price_usd") >= 10) & (F.col("price_usd") < 20))
                     .withColumn("languages", language_band), "languages")
        .orderBy(F.desc("median_reviews")))

,languages,games,median_reviews,breakout_pct,mean_positive_ratio
0,15-20,189,816,42.3,0.827
1,10-14,1028,703,38.8,0.807
2,5-9,1936,264,30.3,0.782
3,21+,99,60,26.3,0.786
4,2-4,2173,57,9.7,0.776
5,1,3597,27,9.1,0.742


The collapse survives the control. Among games priced $10-20, **21 languages or more returns 60
median reviews and a 26.3% breakout rate, against 816 and 42.3% at 15-20 languages** and 703 and
38.8% at 10-14 — it lands just above the 2-4 band. Adding a language string costs nothing and
proves nothing. The band that means something is 10-20, where the localisation is real work.

How far into that band to go is the actual decision, and counting one language at a time answers
it:

In [24]:
display(outcome(games.filter(F.col("n_languages").between(9, 15)), "n_languages")
        .orderBy("n_languages"))

,n_languages,games,median_reviews,breakout_pct,mean_positive_ratio
0,9,1220,175,28.7,0.777
1,10,1043,209,30.8,0.759
2,11,931,281,31.5,0.780
3,12,704,384,38.4,0.776
4,13,509,696,43.2,0.775
5,14,408,299,34.8,0.789
6,15,256,211,31.6,0.784


**Every language added pays up to the thirteenth — 175 median reviews and a 28.7% breakout rate at
nine, 696 and 43.2% at thirteen — and the fourteenth takes it back: 299 and 34.8% over 408
games.** That reversal is the return threshold, and it settles the count on its own.

Which thirteen is a second question, and the catalogue ranking above is the wrong one to answer
it. What matters is the languages the games that actually reached scale ship:

In [25]:
display(games.filter(F.col("owners_min") >= 100000)
             .select(F.explode("languages_list").alias("language"))
             .groupBy("language").agg(F.count("*").alias("breakout_games"))
             .orderBy(F.desc("breakout_games")).limit(15))

,language,breakout_games
0,English,6608
1,German,3634
2,French,3567
3,Spanish - Spain,3310
4,Russian,3014
5,Italian,2749
6,Simplified Chinese,2449
7,Japanese,2250
8,Portuguese - Brazil,1876
9,Polish,1794


The two rankings agree on the set — the same twelve names lead both, reordered, with Spanish and
Italian higher among the games that broke out than in the catalogue at large. The thirteenth is
**Turkish**, which sits fourteenth by catalogue volume and thirteenth here.

**Decision — languages: thirteen — EN, DE, FR, ES, RU, IT, zh-Hans, JA, pt-BR, PL, KO, zh-Hant
and TR.**

## 3.5 Age restriction

In [26]:
display(games.groupBy("age_rating").agg(F.count("*").alias("games")).orderBy(F.desc("games")).limit(8))

,age_rating,games
0,0,55029
1,15,265
2,18,223
3,17,38
4,16,38
5,12,32
6,13,26
7,14,10


Read literally, **only 656 games out of 55 690 (1.2%) carry any age restriction at all**, and 301
are 16+ or over. That is not a description of Steam's catalogue, it is a description of a field
nobody fills in: the store gates mature content through its own content descriptors, and
`required_age` is a legacy attribute left at 0.

The community tags do carry the signal, so the brief's question is better answered with them.

*Chart: bar, `mature` x `breakout_pct`.*

In [27]:
MATURE_TAGS = ["Violent", "Gore", "Nudity", "Sexual Content", "NSFW", "Hentai", "Mature"]

games = games.withColumn("mature",
    F.size(F.array_intersect(F.map_keys("tags"), F.array(*[F.lit(t) for t in MATURE_TAGS]))) > 0)

display(outcome(games, "mature"))
print("mature-tagged games that also declare an age rating:",
      games.filter(F.col("mature") & (F.col("age_rating") > 0)).count(),
      "of", games.filter("mature").count())

,mature,games,median_reviews,breakout_pct,mean_positive_ratio
0,True,7103,71,19.6,0.720
1,False,48587,23,10.8,0.739


mature-tagged games that also declare an age rating: 388 of 7103


**7 103 games — 12.8% of the catalogue — carry mature content tags**, and only 388 of them
declare an age rating for it. They also do better than the rest: median 71 reviews against 23,
and a 19.6% breakout rate against 10.8%.

Caution on the direction: mature themes are concentrated in the kind of large action and RPG
titles that would outperform anyway. The table is not evidence that adding blood sells copies.
It is evidence that **a mature rating is not a commercial handicap on Steam**, which is the only
thing the decision needs.

**Decision — age rating: mature (17+/PEGI 18) is not a constraint on the concept.**

## 3.6 The best rated games

In [28]:
print("games with a 100% positive ratio:", games.filter(F.col("positive_ratio") == 1).count())
display(games.filter(F.col("positive_ratio") == 1)
             .select("name", "positive", "negative", "positive_ratio")
             .orderBy("reviews").limit(5))

games with a 100% positive ratio: 8634


,name,positive,negative,positive_ratio
0,CrossTrix,1,0,1.0
1,Anti-Grav Bamboo-copter,1,0,1.0
2,De Profundis,1,0,1.0
3,Kill Tiger,1,0,1.0
4,The Truck Game,1,0,1.0


Ranking on the raw ratio returns 8 634 games tied at 100%, most of them on one or two reviews.
The Wilson lower bound breaks the tie by asking how much evidence sits behind the score. No
review floor is applied below: thin evidence is exactly what the bound already discounts, and
adding a threshold on top would only hide the fact that it works.

In [29]:
display(games.select("name", "publisher_clean", "reviews",
                     F.round("positive_ratio", 4).alias("positive_ratio"),
                     F.round("wilson_score", 4).alias("wilson_score"),
                     "release_year")
             .orderBy(F.desc("wilson_score")).limit(15))

,name,publisher_clean,reviews,positive_ratio,wilson_score,release_year
0,Flowers -Le volume sur ete-,JAST USA,938,0.9989,0.9940,2018
1,The Void Rains Upon Her Heart,The Hidden Levels,496,1.0000,0.9923,2018
2,Aseprite,Igara Studio,11903,0.9933,0.9916,2016
3,A Short Hike,adamgryu,11732,0.9926,0.9909,2019
4,Senren＊Banka,"HIKARI FIELD, NekoNyan Ltd.",10677,0.9921,0.9903,2020
5,Aventura Copilului Albastru și Urât,Codrin Bradea,2217,0.9937,0.9894,2021
6,祈風 Inorikaze,觀象草圖 Astrolabe Draft,327,1.0000,0.9884,2019
7,People Playground,Studio Minus,144569,0.9886,0.9880,2019
8,Portal 2,Valve,309441,0.9878,0.9874,2011
9,CULTIC,3D Realms,2037,0.9921,0.9873,2022


Two of the 8 634 perfect scores survive it — *The Void Rains Upon Her Heart* on 496 reviews and
*祈風 Inorikaze* on 327 — and the other 8 632 do not. Those two are the largest of the group, which
is the bound working rather than leaking.

The top of the list is not made of blockbusters. *Aseprite* is a pixel-art editor, *A Short Hike*
and *Patrick's Parabox* are one-person indie games, and the only two titles holding that ratio at
scale are **People Playground at 98.9% over 144 569 reviews and Portal 2 at 98.8% over 309 441** —
the more useful benchmark, because sustaining the ratio at that volume is the hard part.

Where Ubisoft's own catalogue sits against that needs the review volume held constant: the
positive ratio climbs with the number of reviews, so comparing Ubisoft to the catalogue at large
would be comparing it to 39 195 games nobody reviewed.

In [30]:
ubisoft = F.col("publisher_clean").rlike("(?i)^ubisoft")

review_rank = (F.when(F.col("reviews") < 10, 0).when(F.col("reviews") < 100, 1)
                .when(F.col("reviews") < 1000, 2).when(F.col("reviews") < 10000, 3).otherwise(4))
review_band = (F.when(review_rank == 0, "1-9").when(review_rank == 1, "10-99")
                .when(review_rank == 2, "100-999").when(review_rank == 3, "1 000-9 999")
                .otherwise("10 000+"))

display(games.filter(F.col("reviews") > 0)
    .withColumn("rank", review_rank).withColumn("reviews_band", review_band)
    .groupBy("rank", "reviews_band").agg(
        F.count("*").alias("games"),
        F.round(F.percentile_approx("positive_ratio", 0.5), 3).alias("median_ratio"),
        F.sum(ubisoft.cast("int")).alias("ubisoft_games"),
        F.round(F.percentile_approx(F.when(ubisoft, F.col("positive_ratio")), 0.5), 3)
         .alias("ubisoft_median"))
    .orderBy("rank").drop("rank"))

,reviews_band,games,median_ratio,ubisoft_games,ubisoft_median
0,1-9,16528,0.800,1,0.800
1,10-99,22667,0.769,7,0.811
2,100-999,11021,0.794,33,0.749
3,1 000-9 999,4104,0.855,55,0.788
4,10 000+,1207,0.894,39,0.830


**Reference point — no Ubisoft title appears in the fifteen above, and once the comparison holds
review volume constant its catalogue sits below the market in every band it occupies: 74.9%
against 79.4% between 100 and 999 reviews, 78.8% against 85.5% between 1 000 and 9 999, 83.0%
against 89.4% above 10 000.** The bar for the next game is not the top of the list — it is
Ubisoft's own back catalogue, and that bar currently sits under the market's.

---
# 4. Genres

`genre` holds one to seven labels per game. Exploding it gives one row per (game, genre), so a
game counted under Action is also counted under RPG — the shares below add up to more than 100%
by construction.

In [31]:
# the raw comma-separated string is dropped: the exploded label replaces it
genre_rows = materialise(games.drop("genre").select("*", F.explode("genres").alias("genre")),
                         "steam_genre_rows")
print(f"{genre_rows.count():,} (game, genre) rows for {games.count():,} games")

157,110 (game, genre) rows for 55,690 games


## 4.1 What is on the shelf

*Chart: bar, `genre` x `games`.*

In [32]:
display(genre_rows.groupBy("genre").agg(F.count("*").alias("games"))
        .withColumn("pct_of_catalogue", F.round(100 * F.col("games") / games.count(), 1))
        .orderBy(F.desc("games")).limit(15))

,genre,games,pct_of_catalogue
0,Indie,39681,71.3
1,Action,23759,42.7
2,Casual,22086,39.7
3,Adventure,21431,38.5
4,Strategy,10895,19.6
5,Simulation,10836,19.5
6,RPG,9534,17.1
7,Early Access,6145,11.0
8,Free to Play,3393,6.1
9,Sports,2666,4.8


**Indie is on 71% of the catalogue** — and it is not a genre. Neither are *Early Access* (11%)
nor *Free to Play* (6%): they describe how a game is funded and sold, not what it is. Excluding
those three, the shelf is **Action (43%), Casual (40%), Adventure (38%), Strategy (20%),
Simulation (19%), RPG (17%)**, and everything else is under 5%.

The field mixes more than that. Its 28 labels cover game genres, funding and release states
(*Indie*, *Early Access*, *Free to Play*), content warnings (*Violent*, *Gore*, *Nudity*), eleven
software categories (*Utilities*, *Photo Editing*, *Design & Illustration*, *Game Development*, …)
and one stray *Movie*. The software rows are software: **Wallpaper Engine, Blender, Aseprite,
Godot Engine and Source Filmmaker** are all in this catalogue because Steam types them as games,
which the `type == 'game'` filter in 2.1 cannot separate. None of those labels reaches 700 games,
but they are not removed either, so they appear in the tables below next to real genres.

For Ubisoft the labels that matter are the six real ones. The rest of section 4 keeps all of
them in the tables, because the contrast between *Indie* and the others is itself informative.

## 4.2 Which genres are liked

Every genre carrying at least 100 games, with no review floor. A floor would lift each genre by
about two points — better-reviewed games are better rated, as 3.6 shows — and drop seven of the
twenty-two, including the lowest-rated one.

*Chart: bar, `genre` x `median_positive_ratio`.*

In [33]:
display(genre_rows.groupBy("genre")
        .agg(F.count("*").alias("games"),
             F.round(F.percentile_approx("positive_ratio", 0.5), 3).alias("median_positive_ratio"),
             F.round(F.sum("positive") / (F.sum("positive") + F.sum("negative")), 3).alias("pooled_ratio"))
        .filter(F.col("games") >= 100).orderBy(F.desc("median_positive_ratio")))

,genre,games,median_positive_ratio,pooled_ratio
0,Game Development,159,0.821,0.893
1,Casual,22086,0.803,0.867
2,Indie,39681,0.800,0.885
3,Adventure,21431,0.797,0.840
4,Action,23759,0.789,0.850
5,RPG,9534,0.779,0.856
6,Strategy,10895,0.769,0.848
7,Design & Illustration,406,0.765,0.961
8,Early Access,6145,0.761,0.822
9,Racing,2155,0.754,0.859


Read on the nine labels that are actually game genres, eight fit inside five and a half points —
**Casual 0.803, Adventure 0.797, Action 0.789, RPG 0.779, Strategy 0.769, Racing 0.754, Sports
0.750, Simulation 0.748** — and the ninth is nowhere near them. **Massively Multiplayer sits at
0.648**, ten points below the lowest of the eight and lowest on the pooled column too at 0.731 —
the only row under it is *Violent*, a content warning rather than a genre. Live-service games are
judged on servers, monetisation and updates long after launch, and this is what that judgement
looks like in aggregate.

The two columns disagree on purpose. The pooled ratio weights every review equally, so it
measures *the average experience across the genre*; the median weights every game equally, so it
measures *the typical game*. Indie is 0.800 typical and 0.885 pooled — its big titles are much
better liked than its median one. The software labels stretch that to absurdity: Photo Editing
reads 0.750 typical against 0.977 pooled, because Wallpaper Engine alone brings 572 127 reviews
to a label of 105 rows.

**Decision — avoid a live-service / MMO structure. The satisfaction penalty is the largest single
effect in the genre data, and the only one attached to a design choice rather than to content or
to a store category.**

## 4.3 Do publishers have favourite genres

*Chart: stacked bar, `publisher_clean` x `games`, grouped by `genre`.*

In [34]:
top_publishers = [r[0] for r in by_publisher.orderBy(F.desc("games")).limit(8).collect()]

display(genre_rows.filter(F.col("publisher_clean").isin(top_publishers))
        .groupBy("publisher_clean", "genre").agg(F.count("*").alias("games"))
        .withColumn("rank", F.row_number().over(
            Window.partitionBy("publisher_clean").orderBy(F.desc("games"))))
        .filter(F.col("rank") <= 3).orderBy("publisher_clean", "rank"))

,publisher_clean,genre,games,rank
0,8floor,Casual,202,1
1,8floor,Strategy,22,2
2,8floor,Simulation,10,3
3,Big Fish Games,Casual,419,1
4,Big Fish Games,Adventure,393,2
5,Big Fish Games,Simulation,7,3
6,Choice of Games,RPG,139,1
7,Choice of Games,Indie,136,2
8,Choice of Games,Adventure,112,3
9,HH-Games,Casual,132,1


Emphatically yes, and the specialisation is near-total: **Big Fish Games is 419 Casual and 393
Adventure out of 423 games**, 8floor is 202 Casual out of 202, Strategy First is Strategy,
Choice of Games is RPG. These are not diversified catalogues — each of these publishers has one
formula and repeats it.

Ubisoft's own Steam catalogue is the same shape, around a different centre:

In [35]:
display(genre_rows.filter(F.col("publisher_clean").rlike("^Ubisoft"))
        .groupBy("genre").agg(F.count("*").alias("games"),
                              F.round(F.sum("owners_x_price") / 1e6).alias("owners_x_price_musd"))
        .orderBy(F.desc("games")).limit(8))

,genre,games,owners_x_price_musd
0,Action,75,3379.0
1,Adventure,49,2298.0
2,Strategy,23,131.0
3,RPG,20,899.0
4,Simulation,18,78.0
5,Racing,14,251.0
6,Casual,13,86.0
7,Indie,7,5.0


**Action (75 titles) and Adventure (49)**, then Strategy and RPG. Whatever the next game
is, it will be read by players against that catalogue.

## 4.4 Which genres are worth entering

`owners_x_price` averaged over each genre, in millions (2.9 defines it). For one game it is every
copy anyone owns, priced at the store's list price and added up; the column averages that over
the genre's games. **It is a total accumulated since release — not a price, and not a yearly
figure.** `mean_price_usd` sits next to it for exactly that reason: what one copy costs, in the
same table as what all the copies add up to. `pct_free` says how much of the genre the proxy
scores at zero.

*Chart: bar, `genre` x `stock_value_musd`.*

In [36]:
display(genre_rows.groupBy("genre")
        .agg(F.count("*").alias("games"),
             F.round(F.avg("owners_x_price") / 1e6, 2).alias("stock_value_musd"),
             F.round(F.avg("initial_price_usd"), 2).alias("mean_price_usd"),
             F.round(100 * F.avg(F.col("is_free").cast("int")), 1).alias("pct_free"))
        .filter(F.col("games") >= 150)
        .orderBy(F.desc("stock_value_musd")))

,genre,games,stock_value_musd,mean_price_usd,pct_free
0,Massively Multiplayer,1460,5.14,5.20,53.0
1,RPG,9534,3.14,9.36,14.6
2,Action,23759,2.63,7.98,13.4
3,Strategy,10895,1.92,8.66,13.9
4,Adventure,21431,1.86,8.31,11.3
5,Simulation,10836,1.80,9.38,12.8
6,Racing,2155,1.28,8.47,12.7
7,Sports,2666,1.19,9.26,15.5
8,Early Access,6145,0.90,8.87,17.9
9,Indie,39681,0.86,6.80,12.4


**Massively Multiplayer leads at 5.14 M$ a game, ahead of RPG at 3.14 M$ and Action at
2.63 M$**, while the two largest shelves in the catalogue carry the least: **Indie 0.86 M$ over
39 681 games, Casual 0.38 M$ over 22 086**.

`mean_price_usd` says where that spread does *not* come from. Across the nine real genres the
average price varies by a factor of **1.8**, from $5.20 for MMO to $9.38 for Simulation, while
the stock value varies by a factor of **13**, from Casual's 0.38 M$ to MMO's 5.14 M$. Choosing a
genre barely moves the price a studio can ask; it moves how many people end up owning the game. The software labels invert the pair: **the six
highest prices in the table are all software, $19.11 to $21.36, and not one of them reaches
0.85 M$**. That is what a niche tool at a high price looks like.

`pct_free` marks the distortion in the other direction. **Free to Play averages 0.03 M$ a game
not because free games make no money but because a list price of zero times any number of owners
is zero** — an in-game economy is invisible here. The same blindness cuts into Massively
Multiplayer, 53% of which is free.

Which is where `mean_price_usd` has to be read a second time, because it averages those zeros
too. For a genre that is half free it reports a share of free games dressed up as a price. The
same three measures on paid games only, where every row has a price a studio could actually set:

*Chart: bar, `genre` x `mean_owners_paid`.*

In [37]:
display(genre_rows.filter(~F.col("is_free")).groupBy("genre")
        .agg(F.count("*").alias("paid_games"),
             F.round(F.avg("initial_price_usd"), 2).alias("mean_price_paid"),
             F.round(F.avg("owners_x_price") / 1e6, 2).alias("stock_value_paid_musd"),
             F.round(F.avg("owners_mid")).alias("mean_owners_paid"))
        .filter(F.col("paid_games") >= 100)
        .orderBy(F.desc("stock_value_paid_musd")))

,genre,paid_games,mean_price_paid,stock_value_paid_musd,mean_owners_paid
0,Massively Multiplayer,686,11.08,10.93,381173.0
1,RPG,8140,10.97,3.67,159757.0
2,Action,20582,9.21,3.04,143310.0
3,Strategy,9386,10.06,2.23,106615.0
4,Adventure,19019,9.36,2.10,103992.0
5,Simulation,9451,10.76,2.06,106276.0
6,Racing,1882,9.70,1.47,68247.0
7,Sports,2253,10.96,1.41,64585.0
8,Animation & Modeling,196,31.40,1.38,206990.0
9,Design & Illustration,286,27.74,1.19,152517.0


**MMO's $5.20 was not a low price, it was a high share of free games** — and the two columns are
related by an identity rather than a tendency: `mean_price_usd` is exactly
`(1 - pct_free) x mean_price_paid`, on every row of the table above to within rounding. It is a
price multiplied by a participation rate, which is why it cannot be read as a price. On its paid
half MMO charges **$11.08, the most of any real genre and 2.1 times what the published column
reports**, and its stock value *rises* to 10.93 M$ instead of falling: the zeros were holding it
down, not propping it up.

The damage is specific, and worth knowing because the rest of section 4 keeps reading that
column. The eight other real genres are **84.5% to 88.7% paid**, so the identity shaves each of
them by about the same eighth and leaves their order untouched — drop MMO and the price spread is
1.62 published against 1.66 paid. **MMO is the one row where half the genre is missing from its
own average**, which is exactly what made it look cheap. Across all nine the spread is **1.7** on
paid games, MMO at the top of it rather than the bottom, while the value spread widens from 13
to **25**.

So the value does come from reach, and here it can be shown rather than inferred: **a paid MMO
averages 381 000 owners against 160 000 for a paid RPG and 143 000 for a paid Action** — two and
a half times the audience, at the same price. What a premium single-player release cannot copy is
that audience and the in-game economy behind it, not a pricing trick.

Set against 4.2, where MMO is last on satisfaction, that leaves **RPG and Action: second and
third on stock value, at ordinary prices, with no satisfaction penalty.**


## 4.5 Is any genre emerging

*Chart: grouped bar, `genre` x `pct`, grouped by `release_year`.*

In [38]:
mix = (genre_rows.filter(F.col("release_year").isin(2017, 2022))
       .groupBy("release_year", "genre").agg(F.count("*").alias("games")))
year_totals = mix.groupBy("release_year").agg(F.sum("games").alias("total"))

genre_mix = (mix.join(year_totals, "release_year")
    .withColumn("pct", F.round(100 * F.col("games") / F.col("total"), 1))
    .groupBy("genre").pivot("release_year", [2017, 2022]).agg(F.first("pct"))
    .withColumnRenamed("2017", "pct_2017").withColumnRenamed("2022", "pct_2022")
    .withColumn("shift_pts", F.round(F.col("pct_2022") - F.col("pct_2017"), 1)))

display(genre_mix.orderBy(F.desc("pct_2022")).limit(12))

,genre,pct_2017,pct_2022,shift_pts
0,Indie,25.1,24.7,-0.4
1,Action,15.6,14.7,-0.9
2,Adventure,13.5,14.5,1.0
3,Casual,14.0,14.5,0.5
4,Strategy,6.4,7.2,0.8
5,Simulation,6.7,7.1,0.4
6,RPG,5.0,6.8,1.8
7,Early Access,3.5,5.7,2.2
8,Sports,2.0,1.5,-0.5
9,Racing,1.3,1.5,0.2


Five years apart, the mix barely moves: Indie 25.1% → 24.7%, Action 15.6% → 14.7%, Casual
14.0% → 14.5%. The only shifts worth naming are **Early Access, 3.5% → 5.7%**, and **Free to Play
collapsing from 2.3% to 0.4%** — the latter partly a labelling change, since free games kept
their share of the catalogue while the genre tag stopped being applied.

**No genre is emerging.** A concept does not need to catch a wave here, because there isn't one;
it needs to be good in a category that already pays.

## 4.6 The slot

Every genre comparison so far ran on the whole catalogue, where a genre's numbers move with what
it charges and how much of it is free — which is exactly what 4.4 had to unpick for MMO. This one
holds both fixed: released from 2018, priced $20-40 — the premium band a Ubisoft release is
written for, and 4.3% of the catalogue (3.3). What is left is
reach and satisfaction, compared on equal commercial ground.

Three of the nine real genres fall below the 150-game floor here — MMO, Racing, Sports — so the
case against live service stays where it was made, in 4.2 and 4.4. Indie and Early Access are in
the table for contrast rather than as candidates, and `is_genre` marks which rows are which.

*Chart: combo — bars `games`, line `breakout_pct`, on `genre`.*

In [39]:
# the band is 3.3's, expressed the same way: [20, 40)
REAL_GENRES = ["Action", "Adventure", "Casual", "RPG", "Strategy",
               "Simulation", "Racing", "Sports", "Massively Multiplayer"]

display(outcome(genre_rows.filter((F.col("release_year") >= 2018)
                                  & (F.col("price_usd") >= 20) & (F.col("price_usd") < 40)), "genre")
        .filter(F.col("games") >= 150)
        .withColumn("is_genre", F.col("genre").isin(REAL_GENRES))
        .orderBy(F.desc("breakout_pct")))

,genre,games,median_reviews,breakout_pct,mean_positive_ratio,is_genre
0,Action,776,622,34.9,0.753,True
1,RPG,471,484,33.8,0.766,True
2,Strategy,474,431,31.2,0.763,True
3,Simulation,550,460,29.3,0.764,True
4,Indie,865,266,28.0,0.783,False
5,Adventure,744,338,27.8,0.790,True
6,Early Access,242,273,27.7,0.764,False
7,Casual,315,77,15.2,0.792,True


In the band Ubisoft would actually price into, **Action and RPG take the top two places, 34.9%
and 33.8% breakout**, ahead of Strategy at 31.2% and Adventure at 27.8%. The same pair leads
4.4's stock value, on a different measure over a different population. That agreement is the
finding.

Their order is not. 34.9% on 776 games and 33.8% on 471 carry standard errors of 1.7 and 2.2
points, so the 1.1-point gap sits well inside them, and Strategy is not separable from RPG
either. **The top three are a plateau, not a ranking.** What does separate is the bottom:
Adventure and Indie are three standard errors under Action, and Casual at 15.2% is in another
regime entirely.

Adventure trades seven points of breakout for the best satisfaction of the three (0.790). Indie's
28.0% is a different animal: 865 of the catalogue's 39 681 Indie rows survive this filter, so the
number does not describe the shelf 4.4 measured at 0.86 M$.

A plateau is an argument for taking both labels rather than choosing between them — provided they
are one position and not two:

In [40]:
slot = games.filter((F.col("release_year") >= 2018)
                    & (F.col("price_usd") >= 20) & (F.col("price_usd") < 40))
a, r = F.array_contains("genres", "Action"), F.array_contains("genres", "RPG")

display(slot.select(
    F.sum(a.cast("int")).alias("action"),
    F.sum(r.cast("int")).alias("rpg"),
    F.sum((a & r).cast("int")).alias("both"),
    F.round(100 * F.sum((a & r).cast("int")) / F.sum(r.cast("int")), 1).alias("pct_of_rpg_also_action")))

,action,rpg,both,pct_of_rpg_also_action
0,776,471,228,48.4


**228 of the band's 471 RPGs also carry Action** — just under half, on the smaller of the two
labels. The rows are not independent samples, and the intersection is a real, populated position
in the catalogue rather than a compromise between two separate genres.

**Decision — genre: Action-RPG, single-player, premium.** Not Casual (15.2%, here). Not live
service (0.648 satisfaction in 4.2, an audience 4.4 shows a premium release cannot buy). Not
Indie-positioned — a funding label rather than a genre (4.1), and not a claim Ubisoft can make.

---
# 5. Platforms

## 5.1 What Steam runs on

*Chart: bar, `platform` x `games`.*

In [41]:
display(games.select(
    F.sum(F.col("windows").cast("int")).alias("windows"),
    F.sum(F.col("mac").cast("int")).alias("mac"),
    F.sum(F.col("linux").cast("int")).alias("linux")))

display(games.groupBy("windows", "mac", "linux").agg(F.count("*").alias("games"))
        .orderBy(F.desc("games")))

,windows,mac,linux
0,55675,12769,8457


,windows,mac,linux,games
0,True,False,False,41271
1,True,True,True,6806
2,True,True,False,5951
3,True,False,True,1647
4,False,True,False,11
5,False,False,True,3
6,False,True,True,1


**Windows is not a choice: 55 675 of 55 690 games support it, and the 15 that do not are
curiosities.** Mac reaches 22.9% of the catalogue and Linux 15.2%, and they travel together —
6 806 games ship all three, more than the 5 951 that add Mac alone.

So the platform question is only ever "do we port", never "which one".

## 5.2 Which genres get ported

*Chart: bar, `genre` x `pct_mac` and `pct_linux`.*

In [42]:
display(genre_rows.groupBy("genre").agg(
            F.count("*").alias("games"),
            F.round(100 * F.avg(F.col("mac").cast("int")), 1).alias("pct_mac"),
            F.round(100 * F.avg(F.col("linux").cast("int")), 1).alias("pct_linux"))
        .filter(F.col("games") >= 150).orderBy(F.desc("pct_mac")))

,genre,games,pct_mac,pct_linux
0,Game Development,159,32.7,22.0
1,Strategy,10895,27.6,16.8
2,Indie,39681,25.0,17.6
3,Free to Play,3393,24.9,14.0
4,Design & Illustration,406,24.6,13.3
5,RPG,9534,23.6,16.0
6,Adventure,21431,23.5,15.4
7,Casual,22086,23.2,15.0
8,Animation & Modeling,322,23.0,11.8
9,Simulation,10836,22.5,14.1


The spread runs from **Strategy at 27.6% Mac and 16.8% Linux down to Early Access at 14.6% and
10.3%**, with Action near the bottom of the range at 19.2% and 14.2%. Turn-based and 2D-heavy
categories port more; action games, which lean hardest on the graphics stack, port least. The
technical cost of the port is visible in the genre mix.

Note the two ends: Strategy is 8 points above Action on Mac, but no genre is anywhere near
Windows' 99.97%.

## 5.3 Does porting pay

*Chart: combo — bars `games`, line `breakout_pct`, on `n_platforms`.*

In [43]:
display(outcome(games, "n_platforms").orderBy("n_platforms"))

,n_platforms,games,median_reviews,breakout_pct,mean_positive_ratio
0,1,41285,21,10.0,0.722
1,2,7599,37,14.0,0.773
2,3,6806,79,21.2,0.783


Three platforms beats one on both measures — 79 median reviews against 21, a 21.2% breakout rate
against 10.0%. That comparison is worthless on its own: a studio that expects a hit is exactly
the studio that pays for ports, so the gap could be entirely selection.

Two checks. First, hold the price band and the release era fixed:

In [44]:
display(outcome(games.filter(F.col("release_year") >= 2018)
                     .withColumn("rank", band_rank)
                     .withColumn("band", price_band)
                     .withColumn("ported", F.col("n_platforms") > 1),
                "rank", "band", "ported")
        .orderBy("rank", "ported").drop("rank"))

,band,ported,games,median_reviews,breakout_pct,mean_positive_ratio
0,free,False,4007,28,12.7,0.697
1,free,True,1219,48,18.0,0.724
2,under $5,False,13769,9,2.0,0.728
3,under $5,True,3315,13,2.9,0.796
4,$5-10,False,6145,12,2.6,0.752
5,$5-10,True,1970,21,4.8,0.813
6,$10-20,False,4911,38,8.9,0.758
7,$10-20,True,1610,87,16.3,0.833
8,$20-40,False,1525,238,25.3,0.764
9,$20-40,True,318,494,31.4,0.809


The gap survives in five of the six bands: at $10-20, 87 median reviews ported against 38; at
$20-40, 494 against 238 and a 31.4% breakout rate against 25.3%. It reverses only above $40, on
48 ported games — too few to read. Second, if porting were a big-publisher behaviour, porting
rates would climb with catalogue size — they do not:

In [45]:
publisher_size = by_publisher.select("publisher_clean", F.col("games").alias("publisher_games"))

display(games.join(publisher_size, "publisher_clean")
        .withColumn("publisher_size", F.when(F.col("publisher_games") == 1, "1 game")
                                       .when(F.col("publisher_games") <= 5, "2-5 games")
                                       .otherwise("6+ games"))
        .groupBy("publisher_size").agg(
            F.count("*").alias("games"),
            F.round(100 * F.avg((F.col("n_platforms") > 1).cast("int")), 1).alias("pct_ported")))

,publisher_size,games,pct_ported
0,2-5 games,15052,27.8
1,6+ games,17486,26.6
2,1 game,23018,24.0


**24.0% for one-game publishers, 27.8% for small ones, 26.6% for large ones** — flat. Porting is
not something only big studios do, which removes the most obvious confounder without removing
the causality problem: within any size class, the games that get ported are still the ones
someone believed in.

What survives is a floor, not a lift: the association is consistent, and the Linux port also
covers the Steam Deck, which shipped in February 2022 and is not yet visible in this snapshot.

**Decision — platforms: Windows at launch, Mac and Linux planned in. Linux is the Steam Deck
route and the cheaper of the two to add once the engine is portable.**

---
# 6. The brief for the next game

Six decisions, each from the section that produced it.

| | decision | evidence |
|---|---|---|
| **Genre** | Action-RPG, single-player, premium | Action and RPG carry 2.63 M$ and 3.14 M$ of stock value a game at ordinary prices, $7.98 and $9.36; 34.9% and 33.8% breakout in the $20-40 band (4.4, 4.6) |
| **Not** | live service or MMO | MMO leads on stock value by audience, not by price: a paid MMO averages 381 000 owners against 160 000 for a paid RPG at the same $11 — an audience, and an in-game economy this dataset cannot see, that a premium release cannot buy. And it carries a 0.648 median positive ratio, ten points below the lowest of the eight other real genres (4.2, 4.4) |
| **Price** | the $20-40 band | a premise this study controls on rather than a finding: 3.3 measures only that the band holds 4.3% of the catalogue and is barely discounted, and 4.6 compares genres inside it. The point within the band is not decided here |
| **Platforms** | Windows at launch, Mac and Linux planned in | Windows is 99.97% of the catalogue; ported games lead inside every price band, and porting is not a big-studio behaviour (5.1, 5.3) |
| **Languages** | thirteen: EN, DE, FR, ES, RU, IT, zh-Hans, JA, pt-BR, PL, KO, zh-Hant, TR | every language added pays up to the thirteenth — 696 median reviews and a 43.2% breakout rate — and the fourteenth reverses it (3.4) |
| **Window** | the first half of the year | the six lightest release months over 2014-2021 are exactly January to June; October ships 4 451 games against January's 3 096 (3.2) |
| **Rating** | mature is not a constraint | 12.8% of the catalogue is mature-tagged and outperforms the rest; Steam's own age field is empty for 98.8% of games (3.5) |

Two things this study says about the field the game will land in, beyond the product itself:

- **The competitor set is in the hundreds, not the fifty thousand.** 41% of Steam's catalogue
  comes from publishers with a single release; the 195 publishers with 21 releases or more hold
  17% of it (3.1). That is the field a Ubisoft release lands in.
- **There is no wave to catch.** The genre mix moved by less than two points in five years. The
  concept has to win on execution inside a category that already pays, not on timing.

The quality bar is section 3.6's: **90% positive is a good Ubisoft-scale result on Steam, and 95%
at scale is exceptional** — Portal 2 territory.

---
# 7. What this dataset cannot decide

**It stops on 11 November 2022.** Every 2022 figure covers ten and a half months, and nothing
after that date exists — no Steam Deck effect, no post-2022 pricing.

**There are no sales.** `owners` is SteamSpy's *estimate*, published as a bracket, and 68% of the
catalogue falls in the bottom one. `owners_x_price` multiplies that bracket's midpoint by the
list price, so it ignores Steam's 30% cut, regional pricing, discounts, refunds, bundles, free
keys — and it values every free-to-play game at zero, which is why section 4.4 reads *Free to
Play: 0.03 M$ a game* for a business model that funds some of the largest games in the table.

**Reviews are not players.** They correlate with owners at 0.76 on the log scale, which is enough
to rank and not enough to size.

**Nothing here is causal.** Price, ports and localisation are all things a studio chooses because
it already expects the game to sell. Section 5.3 removes the most obvious confounder for porting
and cannot remove the rest. Every decision in section 6 is a reading of where successful games
are, not a recipe for becoming one.

**Delisted games are absent.** The catalogue is what was on sale in November 2022, so failures
that were pulled never appear — the breakout rates are, if anything, optimistic.

**One store, one region.** Steam is not consoles, not mobile, not the Epic store, and the prices
are US dollars.